In [2]:
from pathlib import Path
import sys

# Locate project root by searching for the project structure
current = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in [current, *current.parents]
        if (
            (path / "src" / "lithology").is_dir()
            and (path / "data").is_dir()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate ml-lithology-classification project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

Project root: C:\Users\Tammy\Desktop\ml-lithology-classification
Python executable: c:\Users\Tammy\anaconda3\python.exe


In [4]:
from src.lithology.data.las_loader import LASLoader
from src.lithology.preprocessing.quality_assurance import DataQualityAnalyzer

print("Project modules imported successfully.")

Project modules imported successfully.


In [7]:
# Load LAS data and run quality assessment
import pandas as pd
LAS_PATH = PROJECT_ROOT / "data" / "raw" / "1055867994.las"
loader = LASLoader(LAS_PATH)
loader.load()

df = loader.to_dataframe()

analyzer = DataQualityAnalyzer(
    dataframe=df,
    depth_column="DEPTH",
    null_values=[-999.25],
)

report = analyzer.generate_report()

for name, result in report.items():
    print(f"\n{'=' * 50}\n{name.upper()}\n{'=' * 50}")
    if isinstance(result, pd.DataFrame):
        display(result)
    else:
        print(result)

analyzer.export_report(
    PROJECT_ROOT / "outputs" / "reports" / "data_quality"
)


DATASET_SHAPE
(8682, 15)

MISSING_VALUES


,missing_count,missing_percentage
GR,88,1.013591
CNPOR,80,0.921447
RHOB,52,0.598940
RHOC,52,0.598940
DCAL,50,0.575904
DPOR,50,0.575904
PE,50,0.575904
SP,20,0.230362
RILD,10,0.115181
TBHV,5,0.057590



DUPLICATE_RECORDS
0

DUPLICATE_DEPTHS


,DEPTH,GR,TBHV,ABHV,RXORT,SP,RILD,RILM,RLL3,DCAL,RHOB,RHOC,DPOR,PE,CNPOR



INVALID_VALUES


,non_finite,null_sentinels
GR,88,0
TBHV,5,0
ABHV,5,0
RXORT,5,0
SP,20,0
RILD,10,0
RILM,4,0
RLL3,5,0
DCAL,50,0
RHOB,52,0


... (3 outputs left out)

,curve,unique_values,standard_deviation,is_constant,is_near_constant
0,GR,8552,26.114934,False,False
1,TBHV,8632,3877.846363,False,False
2,ABHV,8632,3687.023050,False,False
3,RXORT,8612,66.968767,False,False
4,SP,8628,50.548083,False,False
5,RILD,8003,14410.705461,False,False
6,RILM,7680,21701.336569,False,False
7,RLL3,8006,140.231154,False,False
8,DCAL,4951,10.885369,False,False
9,RHOB,6231,0.483337,False,False



DEPTH_CONTINUITY
{'minimum_depth': 0.0, 'maximum_depth': 4340.5, 'median_step': 0.5, 'mean_step': 0.5, 'minimum_step': 0.5, 'maximum_step': 0.5, 'non_increasing_steps': 0}

STATISTICS


,count,mean,std,min,25%,50%,75%,max
DEPTH,8682.0,2170.250000,1253.210926,0.0000,1085.125000,2170.25000,3255.375000,4340.5000
GR,8594.0,67.429696,26.114934,13.5921,48.399300,66.56070,82.233200,349.7787
TBHV,8677.0,3573.276772,3877.846363,0.0000,358.118400,1057.08890,6727.399900,12230.2520
ABHV,8677.0,3219.119842,3687.023050,0.0000,182.901000,702.94180,6194.322800,11518.1631
RXORT,8677.0,-0.954755,66.968767,-345.3154,-22.713300,-10.66930,-1.738600,385.7625
SP,8662.0,2.126571,50.548083,-178.5555,-26.085250,-4.96485,13.102800,232.6091
RILD,8672.0,2129.504725,14410.705461,0.2948,1.661825,2.81745,6.987750,100000.0000
RILM,8678.0,4962.487081,21701.336569,0.4417,1.471900,2.42775,5.498475,100000.0000
RLL3,8677.0,21.924849,140.231154,0.1653,2.142100,3.66550,7.768100,2024.9390
DCAL,8632.0,20.026798,10.885369,6.1953,8.597150,30.45515,30.487700,59.6927


WindowsPath('C:/Users/Tammy/Desktop/ml-lithology-classification/outputs/reports/data_quality')

In [10]:
from src.lithology.preprocessing.cleaner import DataCleaner

print("Starting preprocessing...")

cleaner = DataCleaner(df, depth_column="DEPTH")

cleaned_df = cleaner.run()

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)
print(cleaner.missing_value_report())

output_path = PROJECT_ROOT / "data" / "interim" / "cleaned_well_logs.csv"
cleaned_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Starting preprocessing...
Original shape: (8682, 15)
Cleaned shape: (8682, 15)
       missing_count  missing_percentage
GR                88            1.013591
CNPOR             80            0.921447
RHOB              52            0.598940
RHOC              52            0.598940
DCAL              50            0.575904
DPOR              50            0.575904
PE                50            0.575904
SP                20            0.230362
RILD              10            0.115181
TBHV               5            0.057590
ABHV               5            0.057590
RXORT              5            0.057590
RLL3               5            0.057590
RILM               4            0.046072
DEPTH              0            0.000000
Saved: C:\Users\Tammy\Desktop\ml-lithology-classification\data\interim\cleaned_well_logs.csv
